<a href="https://colab.research.google.com/github/Roshanraj2580/Rice_Leaf_Detection/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from pathlib import Path
from PIL import Image
import numpy as np
import cv2

# =========================
# Paths
# =========================
input_dir = Path("/content/drive/MyDrive/Minor_Project/Data/Raw/Rice Leaf Disease Image Samples")
output_dir = Path("/content/drive/MyDrive/Minor_Project/Data/Processed")

output_dir.mkdir(parents=True, exist_ok=True)

# =========================
# Settings
# =========================
TARGET_SIZE = 224
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
JPEG_QUALITY = 95

# =========================
# Preprocessing function
# =========================
def preprocess_and_save_image(src_path, dst_path, target_size=224):
    try:
        # Open image and convert to RGB
        img = Image.open(src_path).convert("RGB")
        img = np.array(img)

        h, w = img.shape[:2]

        # Scale image while keeping aspect ratio
        scale = target_size / max(h, w)
        new_w = int(w * scale)
        new_h = int(h * scale)

        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

        # Pad image to make it target_size x target_size
        pad_top = (target_size - new_h) // 2
        pad_bottom = target_size - new_h - pad_top
        pad_left = (target_size - new_w) // 2
        pad_right = target_size - new_w - pad_left

        # Use edge padding to avoid harsh black borders
        padded = cv2.copyMakeBorder(
            resized,
            pad_top, pad_bottom, pad_left, pad_right,
            borderType=cv2.BORDER_REPLICATE
        )

        # Save processed image
        out_img = Image.fromarray(padded)
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        out_img.save(dst_path, quality=JPEG_QUALITY)

        return True

    except Exception as e:
        print(f"Skipped: {src_path} | Error: {e}")
        return False

# =========================
# Process dataset
# =========================
total_saved = 0
total_skipped = 0

class_counts = {}

for class_dir in sorted(input_dir.iterdir()):
    if class_dir.is_dir():
        save_class_dir = output_dir / class_dir.name
        save_class_dir.mkdir(parents=True, exist_ok=True)

        saved_in_class = 0

        for img_path in sorted(class_dir.iterdir()):
            if img_path.is_file() and img_path.suffix.lower() in VALID_EXTS:
                save_path = save_class_dir / img_path.name

                ok = preprocess_and_save_image(img_path, save_path, target_size=TARGET_SIZE)

                if ok:
                    total_saved += 1
                    saved_in_class += 1
                else:
                    total_skipped += 1

        class_counts[class_dir.name] = saved_in_class

print("\nPreprocessing completed.")
print(f"Total saved   : {total_saved}")
print(f"Total skipped : {total_skipped}")
print("\nSaved per class:")
for cls, count in class_counts.items():
    print(f"{cls}: {count}")

print(f"\nProcessed images stored in: {output_dir}")



Preprocessing completed.
Total saved   : 5932
Total skipped : 0

Saved per class:
Bacterialblight: 1584
Blast: 1440
Brownspot: 1600
Tungro: 1308

Processed images stored in: /content/drive/MyDrive/Minor_Project/Data/Processed


In [ ]:
import os
import shutil
import random
from pathlib import Path

# Paths
source_dir = Path("/content/drive/MyDrive/Minor_Project/Data/Processed")
base_output_dir = Path("/content/drive/MyDrive/Minor_Project/Data/Splitted")

train_dir = base_output_dir / "train"
val_dir = base_output_dir / "val"
test_dir = base_output_dir / "test"

# Split ratios
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

random.seed(42)

# Create output folders
for split_dir in [train_dir, val_dir, test_dir]:
    split_dir.mkdir(parents=True, exist_ok=True)

# Process each class folder
for class_dir in source_dir.iterdir():
    if class_dir.is_dir():
        images = list(class_dir.glob("*"))
        random.shuffle(images)

        total = len(images)
        train_end = int(total * train_ratio)
        val_end = train_end + int(total * val_ratio)

        train_images = images[:train_end]
        val_images = images[train_end:val_end]
        test_images = images[val_end:]

        for split_name, split_images in [("train", train_images), ("val", val_images), ("test", test_images)]:
            split_class_dir = base_output_dir / split_name / class_dir.name
            split_class_dir.mkdir(parents=True, exist_ok=True)

            for img_path in split_images:
                shutil.copy(img_path, split_class_dir / img_path.name)

        print(f"{class_dir.name}: Train={len(train_images)}, Val={len(val_images)}, Test={len(test_images)}")

print("Dataset splitting completed.")


Bacterialblight: Train=1108, Val=237, Test=239
Blast: Train=1007, Val=216, Test=217
Brownspot: Train=1120, Val=240, Test=240
Tungro: Train=915, Val=196, Test=197
Dataset splitting completed.
